# 🔬 TEMPO: Semi-Soft Prompt Pool Experiments

**Paper:** TEMPO: Prompt-based Generative Pre-trained Transformer for Time Series Forecasting (ICLR 2024)

**Our Novelty:** Semi-Soft Prompt Pool — semantically initialized prompt pool using GPT-2 word embeddings

**Authors:** Udith V, Sanjai R, Kaviya R, Joshika R | Guide: Dr. P. Kola Sujatha

---

## How to Use This Notebook

1. **Enable GPU:** Runtime → Change runtime type → GPU (T4)
2. **Run cells in order** — each cell has clear instructions
3. **Don't skip cells** — each one depends on the previous

---

## Experiment Overview

| # | Train On | Test On | Prompt Type | Purpose |
|---|----------|---------|-------------|----------|
| 1 | ETTh1 | ETTh1 | Semi-Soft | Baseline (single dataset) |
| 2 | ETTh1 | ETTh1 | **Semi-Soft Pool** | Our novelty (single dataset) |
| 3 | 6 datasets | Weather (unseen) | Semi-Soft | Zero-shot baseline |
| 4 | 6 datasets | Weather (unseen) | **Semi-Soft Pool** | **Our novelty zero-shot** |
| 5 | 6 datasets | ETTm1 (unseen) | Semi-Soft | Zero-shot different target |
| 6 | 6 datasets | ETTm1 (unseen) | **Semi-Soft Pool** | **Our novelty different target** |
| 7 | 6 datasets | Electricity (unseen) | Semi-Soft | Zero-shot electricity |
| 8 | 6 datasets | Electricity (unseen) | **Semi-Soft Pool** | **Our novelty electricity** |
| 9 | 6 datasets | Weather (unseen) | Random Pool | Paper's pool (comparison) |
| 10 | 6 datasets | Weather (unseen) | No Prompt | Ablation (how much do prompts help?) |

## Speed Modes

| Mode | GPT-2 | Data | Epochs | Time/exp | Use For |
|------|-------|------|--------|----------|----------|
| `--fast` | Random | 10% | 3 | ~3-5 min | Code testing |
| `--lite` | Pretrained | 20% | 5 | ~8-12 min | Quick real results |
| `--medium` | Pretrained | 100% | 5 | ~15-20 min | Good results |
| (default) | Pretrained | 100% | 10 | ~30-40 min | Thesis quality |

---
## Step 1: Clone the Repository
This downloads our modified TEMPO code with the Semi-Soft Prompt Pool novelty.

In [ ]:
!git clone https://github.com/sanjai-rs7/TEMPO-SemiSoftPool.git
%cd TEMPO-SemiSoftPool
print('✅ Repository cloned!')

---
## Step 2: Install Dependencies
Installs all required Python packages. Takes ~1-2 minutes.

In [ ]:
!pip install -q transformers peft huggingface_hub omegaconf einops statsmodels reformer_pytorch gdown

# Fix numpy 2.0 compatibility
!sed -i 's/np.Inf/np.inf/g' tempo/utils/tools.py

# Verify GPU
import torch
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('❌ No GPU! Go to Runtime → Change runtime type → GPU')

---
## Step 3: Download All 7 Datasets
Downloads ETTh1, ETTh2, ETTm1, ETTm2, Weather, Electricity, Traffic.

These are the standard time series benchmarks used in the TEMPO paper.

In [ ]:
import urllib.request, os

# --- ETT datasets (from official ETDataset repo) ---
os.makedirs('dataset/ETT-small', exist_ok=True)
base = 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small'
for f in ['ETTh1.csv', 'ETTh2.csv', 'ETTm1.csv', 'ETTm2.csv']:
    urllib.request.urlretrieve(f'{base}/{f}', f'dataset/ETT-small/{f}')
    print(f'✅ Downloaded: {f}')

# --- Weather, Electricity, Traffic (from Google Drive — official Autoformer source) ---
import gdown
print('\nDownloading Weather, Electricity, Traffic from Google Drive...')
print('(This may take 2-5 minutes for large files)')
gdown.download_folder(
    'https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy',
    quiet=False, output='dataset'
)

# Verify all datasets
print('\n--- Downloaded Datasets ---')
!find dataset -name '*.csv' | sort
print('\n✅ All datasets ready!')

---
## Step 4: View the 30 Semantic Prompt Templates (Our Novelty)
These are the English descriptions used to initialize the prompt pool.
Each one is converted to GPT-2 word embeddings as the starting point.

In [ ]:
!python run_experiments.py --exp prompts

---
# 🧪 Running Experiments

Choose ONE of the sections below based on how much time you have.

Each experiment shows:
- Progress bar with iteration speed
- Train/Validation loss per epoch
- Early stopping notifications
- Final MSE and MAE scores
- Comparison table at the end

---
## Option A: ⚡ FAST Mode (~30-50 min for all 10)
- Random GPT-2 weights (no pretrain)
- 10% of training data
- 3 epochs
- **Use for:** Quickly checking that everything works

In [ ]:
# Run ALL 10 experiments in fast mode
!python run_experiments.py --exp all --fast

---
## Option B: 🟢 LITE Mode (~1.5-2 hrs for all 10) ⭐ RECOMMENDED
- **Pretrained GPT-2** (real language knowledge)
- 20% of training data
- 5 epochs
- **Use for:** Getting meaningful results quickly

In [ ]:
# Run ALL 10 experiments in lite mode
!python run_experiments.py --exp all --lite

---
## Option C: 🔶 MEDIUM Mode (~3 hrs for all 10)
- Pretrained GPT-2
- 100% training data
- 5 epochs
- **Use for:** Good quality results

In [ ]:
# Run ALL 10 experiments in medium mode
!python run_experiments.py --exp all --medium

---
## Option D: 🐢 FULL Mode (~6 hrs for all 10)
- Pretrained GPT-2
- 100% training data
- 10 epochs
- **Use for:** Thesis-quality final results

In [ ]:
# Run ALL 10 experiments in full mode
!python run_experiments.py --exp all

---
## Option E: Run Individual Experiments
Pick specific experiments by number. Add any speed mode flag.

In [ ]:
# --- SINGLE DATASET COMPARISON ---
# Exp 1: Baseline semi-soft on ETTh1
# Exp 2: Our Semi-Soft Pool on ETTh1
# Shows: Does our novelty help even on a single dataset?
!python run_experiments.py --exp 1,2 --lite

In [ ]:
# --- THE MOST IMPORTANT COMPARISON (for thesis) ---
# Exp 3: Zero-shot baseline (semi-soft) → Weather
# Exp 4: Zero-shot our novelty (semi-soft pool) → Weather
# Exp 9: Zero-shot random pool (paper's version) → Weather
# Shows: Our novelty vs baseline vs paper's pool on unseen data
!python run_experiments.py --exp 3,4,9 --lite

In [ ]:
# --- ZERO-SHOT ON DIFFERENT TARGETS ---
# Exp 5: Baseline → ETTm1 (unseen)
# Exp 6: Our novelty → ETTm1 (unseen)
# Shows: Does our novelty generalize to a different unseen dataset?
!python run_experiments.py --exp 5,6 --lite

In [ ]:
# --- ZERO-SHOT ON ELECTRICITY ---
# Exp 7: Baseline → Electricity (unseen)
# Exp 8: Our novelty → Electricity (unseen)
# Shows: Does it work on a completely different domain?
!python run_experiments.py --exp 7,8 --lite

In [ ]:
# --- ABLATION: DO PROMPTS MATTER? ---
# Exp 10: No prompt at all → Weather
# Compare with Exp 3 (with prompt) to see how much prompts help
!python run_experiments.py --exp 10,3 --lite

---
# 📊 View Results
Run these cells anytime — even while experiments are still running.

In [ ]:
# View results table with comparisons
# Change the folder name based on which mode you used:
#   experiment_results_fast  (--fast)
#   experiment_results_lite  (--lite)
#   experiment_results_medium (--medium)
#   experiment_results       (default/full)

!python run_experiments.py --exp results

In [ ]:
# Generate and display comparison plots
!python run_experiments.py --exp plots

from IPython.display import Image, display
import os

# Try all possible result directories
for d in ['experiment_results_lite', 'experiment_results_fast', 'experiment_results_medium', 'experiment_results']:
    for p in ['mse_mae_comparison.png', 'baseline_vs_novelty.png']:
        path = f'{d}/plots/{p}'
        if os.path.exists(path):
            print(f'\n📊 {path}')
            display(Image(path))

In [ ]:
# View raw results as JSON
import json, os
for d in ['experiment_results_lite', 'experiment_results_fast', 'experiment_results_medium', 'experiment_results']:
    path = f'{d}/results.json'
    if os.path.exists(path):
        with open(path) as f:
            results = json.load(f)
        print(f'\n📁 {path} ({len(results)} experiments)')
        for r in results:
            status = '✅' if r.get('success') else '❌'
            mse = f"{r['mse']:.4f}" if r.get('mse') else 'N/A'
            mae = f"{r['mae']:.4f}" if r.get('mae') else 'N/A'
            print(f"  {status} {r['name']:<40} MSE={mse}  MAE={mae}  {r.get('time_human','?')}")

---
# 🔄 Updating Code
If the code is updated on GitHub, pull the latest changes:

In [ ]:
!git pull
print('✅ Code updated! Datasets and results are preserved.')

---
# 💾 Save Results to Google Drive
Run this to permanently save your results before the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
save_path = '/content/drive/MyDrive/TEMPO_Results'
os.makedirs(save_path, exist_ok=True)

for d in ['experiment_results', 'experiment_results_fast', 'experiment_results_lite', 'experiment_results_medium']:
    if os.path.exists(d):
        shutil.copytree(d, f'{save_path}/{d}', dirs_exist_ok=True)
        print(f'✅ Saved {d} to Google Drive')

print(f'\n📁 All results saved to: {save_path}')